# Homework 2 Part 2 — Turning clinical text into constraints

**CS224V**

To decide whether a patient qualifies for a clinical trial, someone has to
compare two documents written in prose: the patient's chart and the trial's
eligibility criteria. A common approach today is to give both documents to a
language model and ask it for an answer.

This homework studies a different approach. A language model first
**translates** the criteria and the chart into SMT (satisfiability modulo
theories) — named variables and logical
constraints. A solver then **decides** eligibility from those constraints. You
will see what this makes possible (searching thousands of trials quickly, and
checking *why* a decision was made) and where it falls short.

### Where all of this comes from

Everything in this notebook is a real system, and you can read all of it. Open
these now if you want to follow along in the source; later sections point at
specific files.

| | |
|---|---|
| **the code** | [`stanford-oval/clinical-trial-matching`](https://github.com/stanford-oval/clinical-trial-matching) — start at [`docs/QUICKSTART.md`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/docs/QUICKSTART.md), then [`docs/ARCHITECTURE.md`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/docs/ARCHITECTURE.md) for how the pieces fit |
| **retrieval (Part 2)** | **SatIR**, the system that searches trials with constraints; its paper is [arxiv.org/abs/2604.08849](https://arxiv.org/abs/2604.08849) |
| **the audit (Part 3)** | **VERDICT**, the method for checking why a decision was made; its paper is [arxiv.org/abs/2609.03366](https://arxiv.org/abs/2609.03366) |

You do not need any of it to do the homework. The notebook runs on saved
outputs, and the questions are answerable from what you see here.

### Who does what

Every system in this notebook uses both a language model and ordinary code.
They do different jobs:

| | what it is good at | what it does here |
|---|---|---|
| **the language model** | reading messy prose, e.g. turning "FEV1 < 50% pred" (a lung-function test result) into something named | **translation**: text → named variables and constraints |
| **the code** | giving the same answer for the same input, showing its work, searching 3,621 trials in about three seconds | **decision**: solve the constraints, retrieve, explain, probe |

All of the language-model work is translation, and translation is not a single
call. It is a **chain** of them: the trial text is split into separate
requirements, its medical terms are linked to a standard vocabulary, the
constraints are written piece by piece, and further calls check the earlier
output. The two compilers in the real system use over 160 prompts in total.

### When the model is called

It depends on what is being read:

- **Trials** are translated once, offline, before any patient arrives.
- **Patients, for retrieval** (Part 2), are also read once and on their own,
  into
  constraints over the shared vocabulary. No trial is involved, so this too can
  be done ahead of time.
- **Patients, for a verdict** (Parts 1 and 3), have to be read *against one
  specific trial* — the model is asked about exactly the variables that trial
  declared. That is a **pairwise** call, made when the pair is considered;
  precomputing it would require a model call for every patient–trial pair.

Everything after that read is code: deciding, retrieving, and every audit step
in Part 3.

Whenever you are looking at something a model wrote, the text says so. Watch
that hand-off: the code treats the model's translation as correct, so if the
translation is wrong, the code's results may also be incorrect.

We say *language model* rather than *LLM* because some of the models are not
large: the baseline in Part 2 turns text into vectors with a BERT-sized model,
much smaller than the models that do the translation.

### What you will do

| part | |
|---|---|
| 1 | watch text become constraints, and solve one |
| 2 | turn the constraints into database rows and use them to search for trials |
| 3 | audit a decision: policy, assumptions, pivotal conditions, self-faithfulness |
| 4 | propose how SatIR/VERDICT could improve GRILL from Homework 1 on rare-disease diagnosis |

### Ground rules

- **No API key, no model calls, nothing to install.** Every model step was run
  in advance, and its real output is saved in this notebook. You read those
  outputs instead of calling a model yourself.
- Reflections are graded on reasoning, not on agreeing with us.

## Setup

The cell below is the data: every model output, solver verdict and evaluation
number this notebook uses, compressed into one string so the notebook is a
single
self-contained file. You never need to read it, but run it (and the unpacking
cell after it) before any other code cell: every later cell uses the variables
they define. `extract_artifacts.py` in the
course repo regenerates it from the research pipeline.

In [7]:
# The data, packed. Skip past this one; the next cell unpacks it.
ARTIFACTS = (
    "..."
    "Ic67"
)


In [8]:
# Everything this notebook needs is embedded in the cell below: real outputs
# from real runs, about 187 KB, compressed. No pip install, no API key, no
# download, no files beside this one.
import base64, json, zlib, pathlib, re, collections, itertools, contextlib, io

_BUNDLE = json.loads(zlib.decompress(base64.b64decode(ARTIFACTS)))
PAIRS = _BUNDLE['pairs']
RETRIEVAL = _BUNDLE['retrieval']
POLICY = _BUNDLE['policy']
CATEGORY = _BUNDLE['predicate_categories']
PROJECTION = _BUNDLE['projection']

def clip(text, n):
    """Shorten text for display, marking the cut with an ellipsis."""
    text = text.strip()
    return text if len(text) <= n else text[:n].rstrip() + ' …'

BY_ID = {p['pair_id']: p for p in PAIRS}

# Variable names are long on purpose -- Part 1 explains why -- and they are
# printed in full. Shortening them was tried and abandoned: two different
# variables kept rendering as the same string, which made them impossible to tell apart.
def short(name, width=None):
    return name

def pair_of(p):
    """A pair dict, or the id of one -- so both of these work:

        verdict(PAIRS[0])
        verdict('sigir-20144__NCT02114099')

    Anything else is told what was wanted."""
    if isinstance(p, dict) and 'pair_id' in p:
        return p
    if isinstance(p, str) and p in BY_ID:
        return BY_ID[p]
    if isinstance(p, str):
        raise KeyError(f"no pair {p!r}. Ids look like {PAIRS[0]['pair_id']}")
    raise TypeError(f'expected a pair or a pair id, got {type(p).__name__}. '
                    f'Try PAIRS[0], or a key of BY_ID.')

def readable(line):
    """An assertion without its SMT wrapper or its :named label."""
    body = re.sub(r'\s*:named.*', '', line.strip())
    for prefix in ('(assert (! ', '(assert '):
        if body.startswith(prefix):
            return body[len(prefix):]
    return body

print(f'{len(PAIRS)} patient-trial pairs, already parsed and solved')
print(f"  {sum(1 for p in PAIRS if p['eligible'])} eligible, "
      f"{sum(1 for p in PAIRS if not p['eligible'])} ineligible")
print()
print('they come from the SIGIR clinical-trial collection:')
print(f"  {RETRIEVAL['corpus']['patients']} patients, "
      f"{RETRIEVAL['corpus']['trials']:,} trials")


12 patient-trial pairs, already parsed and solved
  5 eligible, 7 ineligible

they come from the SIGIR clinical-trial collection:
  59 patients, 3,621 trials


---
# Part 1 — Text becomes constraints

Here is one real patient and one real trial. Read both before looking at any
code: everything in Part 1 is about turning these two pieces of prose into
something a solver can check.

In [10]:
# Print the patient note and the trial criteria.
# The case we follow through Parts 1 and 3.
pair = BY_ID['sigir-20144__NCT02114099']

print('PATIENT', pair['patient_id'])
print(clip(pair['patient_note'], 520))
print()
print('TRIAL', pair['trial_id'], '--', clip(pair['trial_title'], 70))
print(clip(pair['trial_criteria'], 620))


PATIENT sigir-20144
A 2-year-old boy is brought to the emergency department by his parents for 5 days of high fever and irritability. The physical exam reveals conjunctivitis, strawberry tongue, inflammation of the hands and feet, desquamation of the skin of the fingers and toes, and cervical lymphadenopathy with the smallest node at 1.5 cm. The abdominal exam demonstrates tenderness and enlarged liver. Laboratory tests report elevated alanine aminotransferase, white blood cell count of 17,580/mm, albumin 2.1 g/dL, C-reactive protein …

TRIAL NCT02114099 -- Trial of Atorvastatin on the Persistent Coronary Aneurysm in Children …
Inclusion criteria: inclusion criteria: 

 Clinical diagnosis of Kawasaki Disease with giant aneurysm 

 Must be older than 10 years old 

 
Exclusion criteria: : 

 Subjects ever received coronary artery bypass graft (CABG) surgery. 

 Subjects have active hepatitis or persistent abnormal liver function such as elevated GOT and GPT. 

 Subjects have the past hi

A person would read both documents and decide. For a solver to decide, the prose
first has to be turned into variables and yes/no conditions. That takes three
steps.

### Step 1 — the language model names things

A model reads the criteria and writes SMT **declarations**: one typed variable
for each thing the trial cares about. This step uses a language model because a
phrase like "FEV1 less than 50% of predicted" has to become a *number with a
unit*, and the model interprets the clinical English to do that.

In the real system, this is a chain of calls, not one: the criteria are split
into separate requirements, each medical term is linked to a standard clinical
vocabulary (so two trials that mean the same thing get the same variable name),
and further calls check that work.

Here is what Step 1 produced for this trial. Each variable is shown with its
type and the meaning the compiler wrote down for it:

In [12]:
# Step 1's output for this trial: the declared variables (language-model output).
decls = [l for l in pair['smt'] if l.lstrip().startswith('(declare')]
meaning = pair['annotations']['variables']
print(f'{len(decls)} variables the trial cares about:\n')
for line in decls:
    name = line.split()[1]
    kind = 'number' if 'Real' in line or 'Int' in line else 'true/false'
    print(f'   {kind:10s} {name}')
    print(f"              means: {meaning.get(name, '(no note)')}")


4 variables the trial cares about:

   true/false patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now
              means: Boolean indicating whether the patient currently has a clinical diagnosis of Kawasaki disease (acute febrile mucocutaneous lymph node syndrome).
   true/false patient_has_diagnosis_of_giant_aneurysm_now
              means: Boolean indicating whether the patient currently has a clinical diagnosis of giant aneurysm.
   true/false patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_inthehistory
              means: Boolean indicating whether the patient has a past clinical diagnosis of Kawasaki disease (acute febrile mucocutaneous lymph node syndrome).
   number     patient_age_value_recorded_now_in_years
              means: Numeric value indicating the patient's current age in years.


The variable names are long on purpose.
`patient_age_value_recorded_now_in_years`
says *whose* attribute, *which* attribute, *when*, and *in what unit*. A short
name like `age` could mean different things in two documents written by
different people; the long name helps clarify the intended attribute in both the
trial and the chart.

### Step 2 — the language model writes the constraints

For simplicity, Part 1 works with the **inclusion criteria only**; exclusion
criteria are handled the same way and we set them aside here. (The verdict
printed in Step 3 does combine both sides.)

Next the model writes **assertions** over those variables — one per
requirement. Each is written in **SMT-LIB**, the standard text format that SMT
solvers such as Z3 read (an open standard maintained by the SMT-LIB initiative
since 2002). Each assertion carries a label attached through `:named`, such as
`REQ0_COMPONENT0`. The labels are what let the solver later report *which*
requirements failed, not just that something did.

Here is what Step 2 produced: each requirement first in the compiler's own
words, then as the SMT-LIB assertion the solver will check.

In [14]:
# Step 2's output for this trial: one assertion per requirement (language-model output).
asserts = [l for l in pair['smt'] if l.lstrip().startswith('(assert')]
said = pair['annotations']['requirements']
print(f'{len(asserts)} requirements:\n')
for line in asserts:
    tag = re.search(r':named (\S+?)\)', line).group(1)
    print(f'   {tag}')
    print(f"      in words : {said.get(tag, '(no note)')}")
    print(f'      in SMT   : {readable(line)}')
    print()


3 requirements:

   REQ0_COMPONENT0_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      in words : To be included, the patient must have a clinical diagnosis of Kawasaki disease.
      in SMT   : (or patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_inthehistory)

   REQ0_COMPONENT1_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      in words : To be included, the patient must have a clinical diagnosis of giant aneurysm.
      in SMT   : patient_has_diagnosis_of_giant_aneurysm_now

   REQ1_COMPONENT0_OTHER_REQUIREMENTS
      in words : To be included, the patient must be age > 10 years.
      in SMT   : (> patient_age_value_recorded_now_in_years 10)



Next to the trial text, the translation can be compared with the criteria. The
criterion
*"Clinical diagnosis of Kawasaki Disease with giant aneurysm"* became two
requirements, `REQ0_COMPONENT0` (Kawasaki disease, now **or** in the past) and
`REQ0_COMPONENT1` (a giant aneurysm). The criterion *"Must be older than 10
years old"* became `REQ1_COMPONENT0`, `(> age 10)`.

What you see here is trimmed for reading. In the compiled file, each declaration
also carries instructions for when the variable is true, false, or unknown, so
the real program is several times longer.

### Reading the chart

A model also reads the patient's chart, asks about exactly the variables Step 1
declared, and records what the chart says. When the chart says nothing, a value
has to come from somewhere else. By **policy** we mean the written rule for that
case: for each kind of variable, what a silent chart should be taken to mean.
Part 3 covers it in detail. Here is what the chart gave for this trial's
variables, and what the policy filled in:

In [16]:
# The chart read (language-model output), plus what the policy fills in.
def policy_value(var, side='inclusion'):
    """What the policy assumes when the chart is silent about var.

    Every pair program in this notebook is inclusion-only, hence the default.
    (Don't take the side from CATEGORY's 'polarity': it is stored once per
    variable name, not per trial, so it can name the wrong side.)"""
    c = CATEGORY.get(var, {})
    rule = POLICY['rules'].get(f"{c.get('level1')}|{c.get('level2')}", {})
    return rule.get(side, 'unknown_defer')

for line in decls:
    name = line.split()[1]
    if name in pair['observed']:
        print(f"   chart says      {name} = {pair['observed'][name]}")
    else:
        print(f'   chart is silent {name}')
        print(f'                   policy: {policy_value(name)}')

   chart says      patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now = True
   chart is silent patient_has_diagnosis_of_giant_aneurysm_now
                   policy: assumed_false
   chart is silent patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_inthehistory
                   policy: assumed_false
   chart says      patient_age_value_recorded_now_in_years = 2.0


So the solver gets: Kawasaki now = `True` and age = `2.0` from the chart, and
*no giant aneurysm* and *no past Kawasaki diagnosis* from the policy
(`assumed_false`: the policy assumes that a condition this rare would have been
written down).

### Step 3 — the solver decides

The language model's part is finished. From here on, everything is code: it
never reads the trial or the chart again, only the variables, the assertions,
and the patient's facts. Together these form an **SMT** problem, and Z3 answers
it: `SAT` if some assignment satisfies every assertion, `UNSAT` if none can.

Here, the answers have a direct meaning. `SAT` means "this patient could
qualify"; `UNSAT` means "these requirements cannot all hold for this patient,"
i.e. the patient is ineligible. The same reading applies to the exclusion
criteria, whose program asserts that the patient is *not* excluded: there,
`UNSAT` means the patient meets an exclusion criterion. For example, this trial
excludes *"Subjects ever received coronary artery bypass graft (CABG) surgery"*,
and its exclusion program asserts `(not
patient_has_undergone_coronary_artery_bypass_graft_inthehistory)`. A patient who
has had a CABG makes that assertion false, so the exclusion side is `UNSAT` and
the patient is excluded.

Before the verdict, here is the check done by hand, one requirement at a time,
with the values above plugged in. (Which requirements fail was computed with Z3
when this notebook's data was prepared; Z3 itself is not installed here.)

In [18]:
# Step 3, traced: plug the patient's values into each requirement.
for line in asserts:
    tag = re.search(r':named (\S+?)\)', line).group(1)
    fails = tag in pair['violated']
    print(f"   {'FAILS   ' if fails else 'can hold'}  {said.get(tag, tag)}")
failing = [t for t in pair['violated']]
print()
if failing:
    print(f'{len(failing)} requirement(s) fail for this patient, so the assertions cannot')
    print('all be true at once: the inclusion side is UNSAT -> INELIGIBLE.')
else:
    print('every requirement can hold: the inclusion side is SAT.')


   can hold  To be included, the patient must have a clinical diagnosis of Kawasaki disease.
   FAILS     To be included, the patient must have a clinical diagnosis of giant aneurysm.
   FAILS     To be included, the patient must be age > 10 years.

2 requirement(s) fail for this patient, so the assertions cannot
all be true at once: the inclusion side is UNSAT -> INELIGIBLE.


Kawasaki can hold (the chart says the child has it now), but the giant aneurysm
fails (the policy assumes none), and the age fails (2 is not over 10). Z3
reaches the same conclusion, and reports it like this:

In [20]:
# The solver's verdict, computed in advance by Z3. No model involved.
def verdict(p):
    """Print what the solver concluded and why; return True if eligible.

    The bool lets you count: with quiet output, sum(verdict(p) for p in
    PAIRS) is how you would score a whole run."""
    p = pair_of(p)
    print(f"{p['pair_id']}")
    print(f"   inclusion criteria : {p['inclusion_status'].upper()}")
    print(f"   exclusion criteria : {p['exclusion_status'].upper()}")
    print(f"   -> {'ELIGIBLE' if p['eligible'] else 'INELIGIBLE'}")
    if p['unsat_core']:
        print(f"   the requirements that could not all hold at once:")
        for name in p['unsat_core']:
            print(f'      {name}')
    return p['eligible']

verdict(pair)


sigir-20144__NCT02114099
   inclusion criteria : UNSAT
   exclusion criteria : SAT
   -> INELIGIBLE
   the requirements that could not all hold at once:
      REQ0_COMPONENT0_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      REQ0_COMPONENT1_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      REQ1_COMPONENT0_OTHER_REQUIREMENTS


That list of requirement names is an **unsat core**: when a set of assertions
has no satisfying assignment, the solver can report a *subset* of them that
already cannot hold together. Here: the Kawasaki requirement, the
giant-aneurysm requirement and the age-over-10 requirement cannot all be true of
this patient.

The core is a by-product of the proof: when asked, the solver records which
assertions it used to prove `UNSAT`, and the labels from Step 2 turn that into
requirement names.

Read it carefully, though. A core is *a* set that conflicts, not a list of what
the patient fails. This one includes the Kawasaki requirement, which the patient
actually meets. Part 3 shows how to get the requirements the patient really
violates. Either way, the decision comes with checkable requirement names; a
free-text matcher may instead return a paragraph.

### The same patient, a different trial

The patient is two years old, so the patient was not eligible for the aneurysm
trial. But
they have Kawasaki disease, and there is another trial for it.

A `SAT` verdict has no core, because there is no conflict. So "why eligible" is
answered differently: for each thing the trial asks about, did the **chart**
answer it, or was it left to the **policy** introduced above? Anything decided
by the policy is marked, because it is not evidence.
The first example below is this same patient; the second is a different patient
and trial, chosen because it has values the chart did not provide.

In [23]:
# For an eligible pair: which requirements did the chart answer, and which were assumed?
def basis(p):
    """For an eligible pair: evidence from the chart vs filled in by policy."""
    p = pair_of(p)
    decl = [l.split()[1] for l in p['smt'] if l.lstrip().startswith('(declare')]
    from_chart = [v for v in decl if v in p['observed']]
    from_policy = [v for v in decl if v not in p['observed']]
    print(f"{p['pair_id']}  ->  {'ELIGIBLE' if p['eligible'] else 'INELIGIBLE'}")
    print(f"   {clip(p['trial_title'], 58)}")
    print(f'   the trial asks about {len(decl)} things')
    for v in from_chart:
        print(f"      CHART   {short(v)} = {p['observed'][v]}")
    for v in from_policy:
        c = CATEGORY.get(v, {})
        where = f"{c.get('level1', '?')} / {c.get('level2', '?')}"
        side = 'inclusion'   # pair programs are inclusion-only, see policy_value()
        rule = POLICY['rules'].get(f"{c.get('level1')}|{c.get('level2')}", {})
        outcome = rule.get(side, 'no rule on file')
        print(f'      POLICY  {short(v)}')
        print(f"                 the chart is silent. It is {c.get('type', '?')}-valued"
              f' and comes from an {side} criterion.')
        print(f'                 classified as:  {where}')
        print(f'                 the rule for that class says:  {outcome.upper()}')
        print(f"                 which means:  {POLICY['states'].get(outcome, '')}")
    if from_policy:
        print(f'   -> {len(from_policy)} of {len(decl)} rest on a rule, not the chart.')
    else:
        print( '   -> every requirement was answered by the chart.')

basis(BY_ID['sigir-20144__NCT00000520'])
print()
basis(BY_ID['sigir-201522__NCT01259336c'])


sigir-20144__NCT00000520  ->  ELIGIBLE
   Prevention of Coronary Aneurysms in Kawasaki Syndrome
   the trial asks about 4 things
      CHART   patient_is_child = True
      CHART   patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now = True
      CHART   patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now@@diagnosed_according_to_centers_for_disease_control_and_prevention_criteria = True
      CHART   patient_presented_to_participating_center_on_or_before_tenth_day_of_illness = True
   -> every requirement was answered by the chart.

sigir-201522__NCT01259336c  ->  ELIGIBLE
   Efficacy Of Itraconazole In Chronic Cavitary Pulmonary Asp …
   the trial asks about 6 things
      CHART   patient_has_finding_of_sputum_culture_positive_for_aspergillus_inthehistory = True
      CHART   patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory@@specimen_is_sputum = True
      CHART   patient_has_finding_of_hyphae_of_aspergillus_detected_i

Both verdicts say `ELIGIBLE`, but they differ in how much recorded information
supports them. The first rests
entirely on the chart (for the things listed above). In the second, half of the
things the trial asks about (three of six) were never recorded, and for each the
policy says `unknown_defer`: it leaves those items unresolved, so the solver may
treat them as
true. That eligibility is therefore *conditional*, and a coordinator should
check those three before contacting the patient.

**Accuracy does not show this difference**: both count as one correct answer.
Part 3 is about making it visible.

> **TODO 1.** Pick any other pair from `PAIRS` and run `verdict` on it. Then
> answer, in a few sentences:
>
> **(a)** Which step would you expect to be the fragile one — the model naming
> variables, the model writing assertions, or the solver? Say why, and what a
> failure of that step would look like in the output above.
>
> **(b)** The text above says a coordinator should check the second pair's three
> unrecorded, deferred items before phoning anyone. Push on that. Look at what
> those three actually are, and say which one you would check *first* and why —
> then say what would have to be true for you to skip the check entirely and
> trust the policy.

*Where this is implemented:* The compilers that produced those variables are
[`trial_compiler/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/trial_compiler)
and
[`patient_compiler/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/patient_compiler);
[`examples/01_decide_a_pair.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/01_decide_a_pair.py)
decides one pair end to end.

In [25]:
# YOUR TURN (TODO 1) -- run verdict() on other pairs and look for yourself.
#
#   for p in PAIRS: print(p['pair_id'], p['eligible'])
#   verdict(BY_ID['sigir-20144__NCT00000520'])   # an eligible one

for p in PAIRS[:3]:
    verdict(p)
    print()


sigir-20144__NCT00000520
   inclusion criteria : SAT
   exclusion criteria : SAT
   -> ELIGIBLE

sigir-20151__NCT02150941a
   inclusion criteria : SAT
   exclusion criteria : SAT
   -> ELIGIBLE

sigir-201522__NCT01259336c
   inclusion criteria : SAT
   exclusion criteria : SAT
   -> ELIGIBLE



In [26]:
# YOUR ANSWERS for TODO 1. Replace the placeholder text, then run the cell.
# Keep the triple quotes. Running it again overwrites the previous answer.

todo_1a = """
The most fragile step is **Step 1 (the language model naming/canonicalizing variables and extracting chart values against them)**, followed closely by **Step 2 (the language model writing SMT assertions)**, whereas **Step 3 (the Z3 SMT solver)** is deterministic, sound, and mathematically exact given its inputs.

**Why Step 1 (and Step 2) are fragile while the solver is not:**
1. **Vocabulary & Canonicalization Drift (Step 1):** Both the trial compiler (`trial_compiler/`) and the patient compiler (`patient_compiler/`) must map free-text clinical prose onto identical canonical predicate names (e.g., mapping "Kawasaki disease" and "acute febrile mucocutaneous lymph node syndrome" both to `patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now`, or preserving exact units and temporal scopes like `patient_age_value_recorded_now_in_years`). If the model fails to canonicalize two synonymous phrasings to the exact same identifier—or mis-extracts a negated/hedged finding from the patient note—the solver treats them as independent variables.
2. **Logical Connective & Polarity Miscompilation (Step 2):** When translating complex prose criteria into SMT-LIB (Cell 13), the model must decide whether commas and clinical lists represent conjunctions (`and`) or disjunctions (`or`), and whether a qualifier (`@@...`) is mandatory or an alternative.

**What a failure looks like in the notebook output:**
- **Step 1 (Variable Naming / Chart Reading) Failure:** In `evidence_states(pair)` (Cell 48) or `basis(p)` (Cell 22), a finding that is clearly documented in the prose patient note would fail to match the trial's declared variable name. Consequently, `name in pair['observed']` would evaluate to `False` (`chart is silent` in Cell 15), and the variable would silently fall back to `policy_value(name)`. For a `1b` ("named specific disease") or `Symptom/Sign / Exam Finding (Current Visit)` predicate whose policy rule is `assumed_false`—such as `patient_has_finding_of_sinus_rhythm_now` in `sigir-20141__NCT01084239` (where the patient note does not explicitly write "sinus rhythm" even if an ECG is normal) or `patient_has_diagnosis_of_giant_aneurysm_now` in `sigir-20144__NCT02114099`—a missed chart extraction turns a chart fact into an `[ASSUMED]` `assumed_false`, causing `verdict(p)` to output a false `UNSAT -> INELIGIBLE` and listing that requirement in `pivotal(p)`.
- **Step 2 (Assertion Writing) Failure:** If the model translates an "at least one of" clinical criterion as `(and ...)` instead of `(or ...)`—for instance, in `sigir-20156__NCT00796913` (*Remission Induction and Sustenance in Graves' Disease 2*), where `REQ0_COMPONENT0` (`patient_has_finding_of_graves_disease_in_remission_after_antithyroid_drug_therapy_now`) and `REQ0_COMPONENT1` (`patient_has_finding_of_hyperthyroidism_in_remission_after_antithyroid_drug_therapy_now`) are asserted as separate conjuncts rather than alternatives—the solver faithfully proves `UNSAT` whenever either phrasing fails, inflating `violated` in `pivotal(p)`.
"""

todo_1b = """
In Cell 22 (`basis(BY_ID['sigir-201522__NCT01259336c'])`, *Efficacy Of Itraconazole In Chronic Cavitary Pulmonary Aspergillosis*), the trial asks about 6 inclusion variables: 3 are answered `True` by the chart (`patient_has_finding_of_sputum_culture_positive_for_aspergillus_inthehistory`, `patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory@@specimen_is_sputum`, and the stem `patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory`), while 3 are silent in the chart and assigned `UNKNOWN_DEFER` by the policy:
1. `patient_has_undergone_aspergillus_fumigatus_1_precipitin_level_inthehistory_outcome_is_positive` (positive *Aspergillus fumigatus* precipitin blood/serology test; `Procedure / Past or Ongoing Diagnostic Procedure`)
2. `patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory@@specimen_is_bronchoalveolar_lavage_fluid` (*Aspergillus* hyphae detected in bronchoalveolar lavage [BAL] fluid; `Symptom/Sign / Other Symptom/Sign`)
3. `patient_has_undergone_aspergillus_species_culture_inthehistory_outcome_is_positive` (positive BAL fluid culture for *Aspergillus* species; `Procedure / Past or Ongoing Diagnostic Procedure`)

**Which one to check first and why:**
If I had to verify one of the three deferred items first, I would check **`patient_has_undergone_aspergillus_fumigatus_1_precipitin_level_inthehistory_outcome_is_positive`** (the serum *Aspergillus* IgG/precipitin test) first. Clinically and operationally, checking or ordering a peripheral blood serology test (precipitins) is fast, inexpensive, and non-invasive, whereas the other two deferred items (`...@@specimen_is_bronchoalveolar_lavage_fluid` and BAL culture) both require an invasive bronchoscopy with bronchoalveolar lavage (BAL)—which is rarely performed if sputum and serology already establish the diagnosis, and carries procedural risk in a 65-year-old patient with hemoptysis ("productive cough with tinges of blood") and a post-tuberculosis cavitary mass.

**Pushing on the premise — what has to be true to skip the check entirely and trust the verdict:**
If we inspect the actual SMT assertion for `sigir-201522__NCT01259336c` (`REQ0_COMPONENT0_OTHER_REQUIREMENTS`), the trial's microbiological criterion (*"Positive results in the aspergillus precipitin test, demonstration of aspergillus hyphae in sputum or BAL fluid or cultures of BAL/sputum growing aspergillus species"*) is compiled as a single 5-way **disjunction (`or`)**:
`(or patient_has_undergone_aspergillus_fumigatus_1_precipitin_level_inthehistory_outcome_is_positive patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory@@specimen_is_sputum patient_has_finding_of_hyphae_of_aspergillus_detected_inthehistory@@specimen_is_bronchoalveolar_lavage_fluid patient_has_undergone_aspergillus_species_culture_inthehistory_outcome_is_positive patient_has_finding_of_sputum_culture_positive_for_aspergillus_inthehistory)`
Because two disjuncts in that `or` clause are **already `explicit_true` from the chart** (`sputum_culture_positive = True` and `hyphae_of_aspergillus_detected...@@specimen_is_sputum = True`), the disjunction `REQ0_COMPONENT0_OTHER_REQUIREMENTS` evaluates to `True` **even if all three `UNKNOWN_DEFER` variables are `False`!**
Therefore, to skip checking the three deferred items entirely, two things must be true:
1. **Logical Sufficiency (Disjunctive Structure):** The trial criterion genuinely requires only *one* of the five microbiological/immunological tests (`or`), as compiled in `REQ0_COMPONENT0_OTHER_REQUIREMENTS`, rather than requiring BAL or precipitins in addition to sputum.
2. **Chart Extraction Fidelity:** We trust the patient compiler's inference that the patient note's morphological description (*"Culture of the sputum revealed an organism with septated, low-angle branching hyphae that had straight, parallel walls"*) is a confirmed *Aspergillus* sputum culture/hyphae finding (rather than an unspeciated fungus that still requires *A. fumigatus* precipitin confirmation).
"""

print('TODO 1: recorded 2 answer(s).')


TODO 1: recorded 2 answer(s).


---
# Part 2 — The same constraints, used for search

Part 1 asked whether one patient is eligible for one trial. In practice, the
question is different: **given one patient and 3,621 trials, which trials are
worth checking?** Running the solver 3,621 times for every patient can be
time-consuming.

So the constraints are turned into database rows that can be searched quickly.
This step is called a **projection**. Two symbols name the before and after:

- **Φ** (phi) is the **original constraint set**: everything the trial's program
  and the patient's record say, in full.
- **Ψ** (psi) is the **projected constraint set**: what remains of Φ after
  dropping the parts a database cannot check.

The projection is **lossy on purpose**. Ψ keeps only conditions that are
*guaranteed* to follow from Φ, so

> **Φ ⇒ Ψ** — every patient–trial pair that Φ would match, Ψ also matches.

This is called **recall-preserving**: Ψ may match pairs that Φ would reject, but
it never misses a pair Φ would accept. In other words, Ψ holds
**necessary conditions, not sufficient ones**. A pair that Ψ rules out is truly
ruled out; a
pair that Ψ matches only goes on a shortlist, and the full solver checks it
against Φ.

**Example.** The aneurysm trial from Part 1 excludes patients with *active*
hepatitis. "Active" is not in the shared vocabulary, so Ψ cannot express this
condition and drops it. A patient with active hepatitis can therefore match Ψ —
a false match, which is allowed — and the solver then rejects them using Φ. The
reverse cannot happen: Ψ never rules out a patient that Φ would accept.

Below, the trial's part of Ψ is called its **gate**, and the patient's part is
called the **patient-side constraints**.

### What gets dropped

The projection drops every variable that uses a **non-canonical** predicate —
wording that was not mapped to the shared vocabulary, so no patient record can
be compared against it. The technique is called **quantifier elimination**: it
removes those variables and keeps everything that still follows without them.
It is applied to trials and to patients separately, before either goes into the
database.

### Three words for the pieces

What remains is a set of requirements, each an OR of simple statements. Three
terms describe these pieces:

- An **atom** is one indivisible statement about one predicate: in Part 1,
  `patient_has_diagnosis_of_giant_aneurysm_now`, or
  `(> patient_age_value_recorded_now_in_years 10)`. Atoms become rows.
- A **clause** is the group of atoms the trial wrote as **one requirement**,
  where the atoms are *alternative* ways of meeting it — any one of them will
  do. `REQ0_COMPONENT0` in Part 1 was a clause holding *Kawasaki now OR Kawasaki
  in the past*: two atoms, either of which satisfies the requirement. That is
  why Level 2 of the search (see "How a search runs" below) marks a clause
  supported as soon as **one** of its atoms
  matches, and why splitting clauses into atom rows loses nothing — the `clause`
  column remembers which requirement the alternatives belong to.
- A **canonical predicate** is the name two atoms must share to be comparable at
  all, and getting there is work in two directions. Across the sides, trial and
  chart were written by different people and must end up in one vocabulary.
  Also *within* each side: one trial can phrase the same condition two ways, one
  chart can record a finding in the narrative and again in a lab table, and
  those must collapse onto one name or the same fact counts twice. That is why
  the Part 1 variable names were so long — the names include details used to map
  two
  phrasings to one name.

### How the projection works

Here is the projection of a real trial from Part 1 (its exclusion side, which
has
more to project). The cell prints the full program and the gate it becomes, so
you can see what survives and what is dropped.

In [30]:
# The real projection, read off a real run. Nothing here is recomputed:
# PROJECTION holds each trial's normalised program and the quantifier-
# eliminated gate that the retrieval database actually stores for it.

side = PROJECTION['NCT02114099']['exclusion']

print('THE FULL EXCLUSION PROGRAM for this trial:')
print(f"   {len(side['variables'])} variables, {len(side['program'])} assertions")
print('   two of those assertions, shortened:')
for a in side['program'][:2]:
    print(f'      {a[:96]}...')

print(f"\nTHE GATE stored for retrieval: {len(side['gate'])} clauses, all of it:")
for a in side['gate']:
    print(f'   {a}')

kept = {v for v in side['variables'] if any(v in g for g in side['gate'])}
gone = [v for v in side['variables'] if v not in kept]
print(f'\n{len(gone)} of the {len(side["variables"])} variables survive into no clause at all:')
for v in gone[:6]:
    print(f'   {v}')
if len(gone) > 6:
    print(f'   ... and {len(gone) - 6} more')


THE FULL EXCLUSION PROGRAM for this trial:
   16 variables, 16 assertions
   two of those assertions, shortened:
      (assert (! (not patient_has_undergone_coronary_artery_bypass_graft_inthehistory) :named REQ0_COM...
      (assert (! (=> patient_has_finding_of_inflammatory_disease_of_liver_now@@active patient_has_find...

THE GATE stored for retrieval: 4 clauses, all of it:
   (assert (not patient_has_undergone_coronary_artery_bypass_graft_inthehistory))
   (assert (not patient_has_diagnosis_of_rhabdomyolysis_inthehistory))
   (assert (or (not patient_has_finding_of_ast_sgot_level_raised_now) (not patient_has_finding_of_alt_sgpt_level_raised_now)))
   (assert (or (not patient_sex_is_female_now) (not patient_is_pregnant_now)))

10 of the 16 variables survive into no clause at all:
   patient_has_finding_of_inflammatory_disease_of_liver_now
   patient_has_finding_of_inflammatory_disease_of_liver_now@@active
   patient_has_finding_of_abnormal_liver_function_now
   patient_has_finding_of

Sixteen assertions over sixteen variables became **four clauses**. The
shrinking is intended.

The variables that disappeared are mostly the ones with `@@` qualifiers, such as
`..._now@@active` and `..._inthehistory@@strong_clinical_support`. (A plain
variable can drop out too:
`patient_has_finding_of_inflammatory_disease_of_liver_now`
appears in this program only alongside its `@@active` refinement, so once the
refinement is dropped nothing is left to test.) The `@@` variables are
**non-canonical** refinements: no patient record is indexed by them, so a
database search could not test them. Quantifier elimination removes them and
keeps only what still follows.

The result is weaker than the program. The program excludes patients with
*active* hepatitis; the gate cannot see "active", so it keeps only what holds
without it. A patient can therefore pass the gate and still be excluded by the
full program.

### The rule, on cases small enough to check by hand

Here is the same operation on small formulas that you can check by hand. Let `a`
and `b` be canonical predicates (a patient record can be indexed by them), and
let `w` be a non-canonical one that must be dropped. The gate keeps a condition
exactly when the program **entails** it — that is, when it must hold no matter
what `w` is.

| the program says | the gate keeps | why |
|---|---|---|
| `a AND w` | `a` | `a` has to hold whatever happens to `w` |
| `a OR b` | `a OR b` | both are canonical, so the alternatives are kept |
| `a OR w` | *nothing* | if `w` might be true, `a` need not be — nothing is guaranteed |
| `w AND (w implies a)` | `a` | `a` follows, even though `w` itself is dropped |

The third row illustrates the effect of a non-canonical alternative. A
requirement that offers a choice between something
testable and something untestable becomes **no constraint at all**. This is
where most of the shrinking comes from. It is also why the gate never rejects a
patient it should keep: dropping a condition can only let *more* patients
through.

### Across all the demo trials

Here is how much each demo trial's program shrinks.

In [33]:
# How much each trial's program shrinks when it is projected.
print(f"{'trial':14s} {'side':10s} {'variables':>10s} {'assertions':>11s} {'gate':>6s}")
tot_a = tot_g = 0
for trial, sides in sorted(PROJECTION.items()):
    for name, d in sorted(sides.items()):
        print(f"{trial:14s} {name:10s} {len(d['variables']):10d} "
              f"{len(d['program']):11d} {len(d['gate']):6d}")
        tot_a += len(d['program']); tot_g += len(d['gate'])
print(f'\n{tot_a} assertions in total become {tot_g} gate clauses.')
zero = sum(1 for v in PROJECTION.values() for d in v.values() if not d['gate'])
sides = sum(len(v) for v in PROJECTION.values())
print(f'{zero} of the {sides} sides project to nothing at all: for those the gate')
print('cannot rule anybody out, so every patient passes and the solver decides.')


trial          side        variables  assertions   gate
NCT00000520    exclusion           3           2      0
NCT00000520    inclusion           6           2      1
NCT00233194a   exclusion          43          35      8
NCT00233194a   inclusion           7           3      0
NCT00393913    exclusion          20          20      8
NCT00393913    inclusion           6           4      1
NCT00774137a   exclusion          14          13      1
NCT00774137a   inclusion          13           3      0
NCT00796913    exclusion           4           5      1
NCT00796913    inclusion           2           2      0
NCT00851487    exclusion          56          47     14
NCT00851487    inclusion           8           8      0
NCT00929721    exclusion           6           6      1
NCT00929721    inclusion           4           4      1
NCT01084239    exclusion          53          42     10
NCT01084239    inclusion          30           9      1
NCT01259336c   exclusion          12          11

Some sides project to **zero** clauses (8 of the 24 in the table above): none of
their logic can be expressed in the shared vocabulary, so the gate cannot
exclude anyone. That is still recall-preserving; it just saves no work for that
trial.

The gate has a second limit. Each clause is checked on its own, so **a set of
rows cannot express that one atom rules another out**. Nothing stops a patient
from matching "under five" in one clause and "over ten" in another, or a finding
from being treated as both present and absent.

Both limits are associated with the gate's speed-oriented design. They are
covered by running the full solver
afterward on every trial the gate lets through — which is why `NCT02114099` is
retrieved and only then found `INELIGIBLE`.

> **TODO 2.** Reflect on the projection you just watched:
>
> **(a)** The gate is a set of independent rows, so it cannot record that one
> atom rules another out — nothing stops one clause being supported by *age
> under 5* while another is supported by *age over 10*. Describe a concrete pair
> of requirements that the gate would let through together but no real patient
> could satisfy.

In [36]:
# YOUR ANSWERS for TODO 2. Replace the placeholder text, then run the cell.
# Keep the triple quotes. Running it again overwrites the previous answer.

todo_2a = """
Because the SatIR retrieval gate (Part 2, Cells 26–33) projects a trial's SMT program into a set of **independent clause rows** and checks each clause in isolation (marking a clause supported as soon as at least one of its atoms is not contradicted), it cannot enforce cross-clause mutual exclusion or functional dependencies across variables.

**Concrete examples of mutually incompatible requirements that the gate lets through together:**
1. **Cross-Clause Numeric / Temporal Contradiction (e.g., Age Bounds in `NCT01084239` or Pediatric vs. Geriatric Clauses):**
   Suppose Clause 1 in a trial gate requires `(or (< patient_age_value_recorded_now_in_years 5) patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now)` (satisfied via the Kawasaki atom or an unindexed atom) and Clause 2 requires `(> patient_age_value_recorded_now_in_years 10)` (as in `NCT02114099`'s `REQ1_COMPONENT0_OTHER_REQUIREMENTS`), or even more starkly, Clause 1 is `(< patient_age_value_recorded_now_in_years 18)` and Clause 2 is `(>= patient_age_value_recorded_now_in_years 50.0)` (as in `NCT00929721`). If a patient's chart is silent on age—or if each clause is matched via a different disjunct—the gate checks Clause 1 alone (satisfiable by picking `age = 2`) and Clause 2 alone (satisfiable by picking `age = 55`). Because the database join does not require a single simultaneous assignment to `patient_age_value_recorded_now_in_years` across rows, the gate lets the trial through, even though $\text{age} < 18 \wedge \text{age} \ge 50$ is unsatisfiable for any real patient.
2. **Simultaneous Presence and Absence of the Same Clinical State Across Inclusion and Exclusion Gates (or Split Clauses):**
   Consider a trial whose inclusion gate requires active hyperthyroidism/Graves' disease (`patient_has_finding_of_hyperthyroidism_now = True`) while another clause (or exclusion gate clause) requires `(not patient_has_finding_of_hyperthyroidism_now)`—or a trial requiring both `(not patient_sex_is_female_now)` in one clause and `patient_is_pregnant_now = True` in another (compare `NCT02114099`'s exclusion gate clause `(or (not patient_sex_is_female_now) (not patient_is_pregnant_now))` in Cell 29). Or consider `NCT02114099` itself: its inclusion side projects to **0 gate clauses** (Cell 32: `NCT02114099 inclusion: 5 variables, 3 assertions -> 0 gate clauses`) while its exclusion side projects to 4 clauses. A 2-year-old patient (`sigir-20144`) surfaces via the patient-side disease key `patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now` (Cell 42) and passes all 4 exclusion gate clauses, only for the full Z3 solver to reject the pair on `(> patient_age_value_recorded_now_in_years 10)` and `patient_has_diagnosis_of_giant_aneurysm_now`.
"""

print('TODO 2: recorded 1 answer(s).')


TODO 2: recorded 1 answer(s).


### How a search runs

Once trials and patients are both stored as rows, a search is three database
joins:

| level | what gets joined | what comes out |
|---|---|---|
| **1. atom** | trial atoms ↔ patient atoms sharing a canonical predicate | compatible atom pairs |
| **2. clause** | atom pairs ↔ the clause each belongs to | a clause is *supported* if ≥1 of its atoms matches |
| **3. trial** | supported clauses back up to the trial | the trial is returned only if every retrieval-relevant clause is supported |

No model is called during a search; all model work happened earlier, when each
document was indexed. **Search runs without model calls**, and it takes about
three
seconds across all 3,621 trials.

### What it is being compared against

The baseline works in two steps. A language model reads the patient note and
writes a short list of **keywords** (the conditions and findings worth searching
on). Those keywords are turned into a vector, and so is each trial. Retrieval
returns the trials whose vectors are closest.

Both approaches use language models, and both do that work ahead of time; at
search time, both are pure code. So this is not "with a model" versus "without".
The difference is *what the model is asked to write down*:

| | what the model writes down | what the query does with it |
|---|---|---|
| **embedding baseline** | keywords, then a vector of them | nearest neighbours by distance |
| **the constraint side** | named variables and constraints | a join on shared atoms |

That difference is what the comparison tests. A vector is a single point: it
does not explicitly identify *which* keyword matched, and the keyword list may
not preserve
whether a finding was present or absent. Named constraints keep that
information, which is why a join can report the reason for each match.

In [39]:
# Recall for each retrieval objective, from the evaluation on the
# full corpus. 'Recall' = of the trials a clinician judged useful for this
# patient, what fraction did the system actually return?
print(f"{RETRIEVAL['corpus']['patients']} patients, "
      f"{RETRIEVAL['corpus']['trials']:,} trials\n")
print(f"{'objective':30s} {'constraints':>12s} {'embeddings':>11s} {'gap':>7s}")
for o in RETRIEVAL['objectives']:
    gap = o['satir_recall'] - o['embedding_recall']
    print(f"{o['name']:30s} {o['satir_recall']:11.1f}% {o['embedding_recall']:10.1f}%"
          f" {gap:+6.1f}")
print()
print(f"query time, constraint side: "
      f"{RETRIEVAL['query_seconds_per_patient_satir']}s per patient "
      f"over {RETRIEVAL['corpus']['trials']:,} trials")


59 patients, 3,621 trials

objective                       constraints  embeddings     gap
treat the chief complaint             93.7%       62.4%  +31.2
treat any condition                   92.1%       53.7%  +38.4
relevant to any condition             92.5%       70.3%  +22.3

query time, constraint side: 2.95s per patient over 3,621 trials


**Recall** is the fraction of the trials a clinician judged useful for a patient
that the system actually returned. The constraint side's recall is 20 to 40
points higher. Two things about how recall was measured matter here:

- Recall is measured against **what a clinician judged useful**, not against a
  text-similarity score. A system optimized for text similarity is not being
  scored on its own objective.
- There are **three different objectives** ("treat the chief complaint", "treat
  any condition", "relevant to any condition"), and the constraint side can be
  *told which one you mean*. The embedding side does not explicitly encode that
  choice: the vector representation does not include a dedicated field for
  "I want trials that treat this, not merely mention it."

The second point may explain much of the gap, so it does not show that
constraints
are simply better than embeddings.

In [41]:
# How many trials each system hands back per patient. Recall is
# high if you return everything, so always look at this alongside it.
print(f"{'objective':30s} {'constraints':>12s} {'embeddings':>11s}")
for o in RETRIEVAL['objectives']:
    print(f"{o['name']:30s} {o['satir_trials_per_patient']:11.2f} "
          f"{o['embedding_trials_per_patient']:10.2f}")
print()
s = RETRIEVAL['patients_served_with_at_least_one_useful_trial']
print(f"patients given at least one useful trial: "
      f"{s['satir']} vs {s['embedding']} of {RETRIEVAL['corpus']['patients']}")


objective                       constraints  embeddings
treat the chief complaint             3.25       2.17
treat any condition                   5.12       2.98
relevant to any condition            11.76       8.93

patients given at least one useful trial: 54 vs 50 of 59


### Patient-side constraints

The patient's record is projected just as a trial's is, into **patient-side
constraints** over the same canonical predicates — the yes/no findings the
chart records as present. Each says how the patient relates to the fact: a
`disease`
recorded as present, or some other `positive finding`. These are what retrieval
starts from: for each patient-side constraint, which trials are aimed at it?

**Example.** For the 2-year-old from Part 1, the patient-side constraints
include *has Kawasaki disease now* (a `disease`), and *is a child* and
*presented by day ten of the illness* (both `positive finding`s).

The cell below runs that join by hand. Every trial it lists comes with the
patient-side constraints that brought it in, so "why was this retrieved" has an
answer identifying the patient-side constraints used in retrieval.

One caveat: this notebook does not include the patient's standalone retrieval
read, so the cell uses the facts from this patient's pairwise read against
`NCT00000520` as a stand-in. Those facts answer exactly what that trial asked,
so it is bound to surface; the informative part is *which* other trials do.

In [43]:
# Level 1 of the join, on the pairs you already have.
#
# A patient's recorded facts are p['observed']; what a trial asks about is
# its declared variables. Matching those is the atom level.
#
# Three restrictions below are not simplifications -- they are what the real
# join does, and leaving any of them out changes the answer:
#
#   1. only BOOLEAN facts, and only those recorded TRUE, can be matched.
#      The engine joins on `kind='bool'` and a truthy value. A number like
#      age is never a key: every patient has an age and every trial asks
#      about it, so it would match everything and discriminate nothing.
#      Numbers are constraints to CHECK later, not keys to retrieve on.
#   2. each patient-side constraint is TYPED: the patient is linked to it by a
#      relation, and the engine keeps them apart: a disease the patient is
#      recorded as having, versus any other positive finding. The real
#      system also canonicalises names before comparing them.
#   3. no @@ refinements. They are non-canonical (see "What gets dropped"
#      in the projection section), so the index has no rows for them.

def hit_kind(var):
    """How is the patient related to this fact? Disease, or other finding."""
    level1 = CATEGORY.get(var, {}).get('level1')
    return 'disease' if level1 == 'Disease/Disorder' else 'positive finding'

def is_key(var, value):
    """Usable as a retrieval key: boolean, recorded True, and canonical."""
    return value is True and '@@' not in var

def shared_facts(patient_pair, trial_pair):
    """The atoms that let this trial surface, each with its relation type.

    Boolean, recorded True, and canonical only -- see the restrictions above."""
    patient_pair, trial_pair = pair_of(patient_pair), pair_of(trial_pair)
    asked = {l.split()[1] for l in trial_pair['smt']
             if l.lstrip().startswith('(declare')}
    return sorted((hit_kind(v), v) for v in asked & set(patient_pair['observed'])
                  if is_key(v, patient_pair['observed'][v]))

me = BY_ID['sigir-20144__NCT00000520']
usable = [v for v, val in me['observed'].items() if is_key(v, val)]
print(f"patient {me['patient_id']}: {len(me['observed'])} recorded facts, "
      f"{len(usable)} of them usable as retrieval keys")
print('the rest are numbers, recorded False, or @@ refinements, and are not used as keys:')
for v, val in me['observed'].items():
    if not is_key(v, val):
        if not isinstance(val, bool):
            why = 'a number'
        elif val is False:
            why = 'recorded False'
        else:
            why = 'an @@ refinement, not canonical'
        print(f"      {v} = {val}  ({why})")
print()
missed = []
for other in PAIRS:
    shared = shared_facts(me, other)
    if shared:
        print(f"{other['trial_id']} would surface, because this patient has:")
        for kind, v in shared:
            print(f"      [{kind:16s}]  {v}")
    else:
        missed.append(other['trial_id'])
print(f"\nno shared atom, so never even looked at: {len(missed)} trials")
print('   ' + ', '.join(missed))

patient sigir-20144: 6 recorded facts, 3 of them usable as retrieval keys
the rest are numbers, recorded False, or @@ refinements, and are not used as keys:
      exclusion_presented_after_tenth_day_of_illness = False  (recorded False)
      patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now@@diagnosed_according_to_centers_for_disease_control_and_prevention_criteria = True  (an @@ refinement, not canonical)
      patient_time_of_illness_onset_value_recorded_at_presentation_withunit_days = 5.0  (a number)

NCT00000520 would surface, because this patient has:
      [disease         ]  patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now
      [positive finding]  patient_is_child
      [positive finding]  patient_presented_to_participating_center_on_or_before_tenth_day_of_illness
NCT02114099 would surface, because this patient has:
      [disease         ]  patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now

no sh

### Retrieval uses both sides

That is the patient side. Retrieval applies **both**:

- the **patient-side constraints** bring trials in: a trial is a candidate
  because it targets something the patient has;
- the **trial-side constraints** — the gate from the projection section — then
  rule candidates out if the patient's facts violate one of the gate's
  clauses, which every eligible patient must satisfy.

So the explanation has two halves: *why it was found* (the patient-side
constraints above) and *why it was not ruled out* (the gate clauses it passed).
A vector similarity score of `0.83` alone does not provide either explanation.

The two types are kept separate rather than counted as one "overlap", because
they license different things: a shared diagnosis is a reason to consider a
trial, but a shared incidental finding usually is not. `NCT02114099` comes in on
the Kawasaki diagnosis — it is a Kawasaki trial — and Part 1 showed
the solver finds this patient `INELIGIBLE` for it. That is not a retrieval
failure. Retrieval's job is to produce a short list worth checking; the solver's
job is to decide.

*Where this is implemented:* The joins are implemented in
[`sql_retrieval/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/sql_retrieval)
(its `README.md` provides a description), and
[`examples/06_satir_retrieval.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/06_satir_retrieval.py)
runs a retrieval. The argument is in the SatIR paper,
[arxiv.org/abs/2604.08849](https://arxiv.org/abs/2604.08849).

---
# Part 3 — Auditing a decision

The audit provides information beyond the verdict. A coordinator told
"ineligible" needs to know what
the decision rests on, what the system assumed, and what would change it.

This part takes one decision through four questions. **All four are answered by
code**, not by the language model: the model's work (translating the criteria
and reading the chart) is already done, and the audit only examines the
solver's work.

| question | what it asks | who answers |
|---|---|---|
| **policy** | the chart is silent here — what do we do about that? | code, reading a written-down policy |
| **assumptions** | what did we fill in that nobody actually recorded? | code |
| **pivotal** | which conditions is this verdict actually resting on? | code, reading the solver's output |
| **self-faithful** | if those conditions changed, would the verdict change? | code, by re-running |

**A note on terms.** A *variable* is one declared name. A *requirement* is one
named assertion from the trial's criteria; helper assertions the compiler adds
(named `..._AUXILIARY...`, e.g. "a qualified finding implies the plain one") are
definitions, not requirements. The *pivotal conditions* are the requirements the
patient's evidence actually violates. As in Part 1, the audit below works on
each pair's **inclusion** requirements only, so counts of chart facts versus
assumptions cover those; assumptions on the exclusion side are not listed.

### 1. Policy — what silence means

Charts are incomplete. A trial requires "no history of multiple sclerosis" and
the chart does not mention MS. There are three options: assume the
patient does not have it, assume they might, or refuse to decide and pass it to
a person.

This choice can affect matcher reliability, and its effects may go unnoticed. So
it is not left to the model or to whoever wrote the code. It is a
**written policy** that says, for each kind of predicate, what silence means.

Numbers need one extra step, because "the chart is silent about age" cannot be
read as true or false — a number is not a yes/no. So a numeric predicate is
**curried** first: the threshold and the condition on it are folded into a
single yes/no predicate. `age > 10` stops being *a number and a comparison* and
becomes one boolean, *age is over ten*, and that boolean gets an evidence state
like any other.

In [47]:
# The policy is a fixed, versioned document, not a prompt to a model.
print(f"policy: a fixed table of {len(POLICY['rules'])} rules, one per kind of predicate\n")
print('the five states a predicate can be in:')
for state, meaning in POLICY['states'].items():
    print(f'   {state:16s} {meaning}')
print()
print('what silence means depends on the KIND of predicate:')
for c in POLICY['categories']:
    print(f"   {c['id']}  {c['what']:26s} silent -> {c['silent_means']}")
    print(f"       e.g. {', '.join(c['examples'][:3])}")
    print(f"       why: {c['why']}")


policy: a fixed table of 29 rules, one per kind of predicate

the five states a predicate can be in:
   assumed_false    chart is silent; policy reads silence as false
   assumed_true     chart is silent; policy reads silence as true (rare)
   explicit_false   the chart documents that this does NOT hold
   explicit_true    the chart documents that this holds
   unknown_defer    chart is silent; policy refuses to guess, defer to the visit

what silence means depends on the KIND of predicate:
   1a  common chronic disease     silent -> unknown_defer
       e.g. hypertension, type 2 diabetes, asthma
       why: common enough that a silent chart is weak evidence either way
   1b  named specific disease     silent -> assumed_false
       e.g. cystic fibrosis, multiple sclerosis, end-stage renal disease
       why: a condition this salient would have been written down
   1c  recent acute event         silent -> assumed_false
       e.g. MI within 30 days, acute exacerbation
       why: a rec

Silence is interpreted differently depending on the condition. A chart that does
not mention **asthma** tells you little — many people have mild asthma that is
not in the note — so the policy says `unknown_defer`. A chart that does not
mention **cystic fibrosis** is treated as informative under this policy, because
the policy assumes that a chart for a CF patient would mention it, so the policy
says `assumed_false`.

That reflects the policy's clinical assumptions. It is written down **once** by
a person and applied
the same way to every pair, instead of having a model decide again, and possibly
differently, each time.

Here is one case of each state from the demo pairs:

| state | example |
|---|---|
| `explicit_true` | the chart says the 2-year-old has Kawasaki disease now |
| `explicit_false` | the chart for patient `sigir-201522` records that the patient is not pregnant |
| `assumed_false` | the 2-year-old's chart never mentions a giant aneurysm; it is a rare, specialist-managed disease that the policy assumes would be recorded, so silence is read as "no" |
| `unknown_defer` | the aspergillosis patient's chart never mentions an *Aspergillus fumigatus* precipitin test; for that class of diagnostic test the policy leaves the status unresolved |
| `assumed_true` | rarely used: silence is read as "yes" |

### 2. Assumptions — what we filled in

Whenever the policy fills in a value that the chart did not provide, that value
is an **assumption**. The system lists every one, because these are the parts of
the verdict that rest on a rule rather than on evidence, so a reviewer can
check them first.

In [49]:
# `observed` is what the pipeline actually read off the chart.
# Anything the trial asks about that is NOT in there is an assumption.
def evidence_states(p):
    """Per variable: did the chart say, or is it an assumption?"""
    p = pair_of(p)
    rows = []
    for line in p['smt']:
        if not line.lstrip().startswith('(declare'):
            continue
        var = line.split()[1]
        cat = CATEGORY.get(var, {})
        rows.append(dict(var=var,
                         observed=var in p['observed'],
                         value=p['observed'].get(var),
                         kind=cat.get('level2', 'uncategorised'),
                         why=cat.get('rationale', '')))
    return rows

rows = evidence_states(pair)
print(pair['pair_id'])
print()
for r in rows:
    state = None if r['observed'] else policy_value(r['var'])
    mark = 'CHART   ' if r['observed'] else ('DEFERRED' if state == 'unknown_defer' else 'ASSUMED ')
    val = f" = {r['value']}" if r['observed'] else ''
    print(f"  [{mark}] {short(r['var'])}{val}")
    print(f"           kind: {r['kind']}")
    if not r['observed'] and r['why']:
        print(f"           policy: {state} -- {r['why'][:70]}")
print()
n_ass = sum(1 for r in rows if not r['observed'] and policy_value(r['var']) != 'unknown_defer')
n_def = sum(1 for r in rows if not r['observed'] and policy_value(r['var']) == 'unknown_defer')
print(f'{n_ass} of {len(rows)} are assumptions (a value the policy filled in);')
print(f'{n_def} are deferred (left open); the rest are chart facts.')


sigir-20144__NCT02114099

  [CHART   ] patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_now = True
           kind: Uncommon Acute Disease
  [ASSUMED ] patient_has_diagnosis_of_giant_aneurysm_now
           kind: Uncommon Chronic Disease
           policy: assumed_false -- Giant aneurysm is rare and specialist-managed.
  [ASSUMED ] patient_has_diagnosis_of_acute_febrile_mucocutaneous_lymph_node_syndrome_inthehistory
           kind: Uncommon Acute Disease
           policy: assumed_false -- Kawasaki disease (the syndrome) is rare and acute.
  [CHART   ] patient_age_value_recorded_now_in_years = 2.0
           kind: Age/Sex

2 of 4 are assumptions (a value the policy filled in);
0 are deferred (left open); the rest are chart facts.


### 3. Pivotal conditions — what the verdict rests on

Part 1 showed the unsat core. Z3 returns *a* subset of the assertions that
cannot
hold together — not necessarily the smallest one, and not the only one. A
different random seed or assertion order can give a different core.

So the core is not "the reasons". It can even name a requirement the patient
*meets*: the Kawasaki trial's core included the Kawasaki requirement, and this
patient has Kawasaki disease.

What VERDICT reports instead are the **pivotal conditions**: every requirement
the patient's evidence actually **violates** — each checked against the chart
facts plus the policy's assumptions. Fixing **all** of them is necessary for the
decision to flip: if a patient violates three requirements, fixing one or two
changes nothing — each remaining violation is enough on its own to keep them
ineligible.

**Example.** For the 2-year-old and the aneurysm trial, two requirements are
violated: *older than 10* (the chart says 2) and *giant aneurysm* (the chart is
silent, and the policy assumes no). Those two are the pivotal conditions. Fixing
only the age still leaves no aneurysm, so the patient stays ineligible; only
fixing both flips the verdict. The unsat core, by contrast, also named the
Kawasaki requirement, which the patient meets.

We found them with Z3 when preparing this notebook's data and saved the result
with each pair.

In [51]:
# A bigger trial: how many of its requirements does this patient violate?
big = BY_ID['sigir-20141__NCT01084239']
named = {re.search(r':named (\S+?)\)', l).group(1)
         for l in big['smt'] if ':named' in l}
reqs = {n for n in named if 'AUXILIARY' not in n}   # helpers are definitions
print(clip(big['trial_title'], 66))
print(f'   requirements in this trial : {len(reqs)}')
print(f"   violated by this patient   : {len(big['violated'])}")
print()
n = len(big['violated'])
print(f"   {'both' if n == 2 else f'all {n}'} must be fixed for the verdict to flip.")


Multicenter Study to Rule Out Myocardial Infarction by Cardiac Com …
   requirements in this trial : 6
   violated by this patient   : 2

   both must be fixed for the verdict to flip.


In [52]:
# Every requirement the patient's evidence violates, in readable form.
def pivotal(p):
    """The pivotal conditions: requirements the evidence actually violates."""
    p = pair_of(p)
    if p['eligible']:
        print(f"{p['pair_id']}: eligible -- nothing is violated.")
        return []
    by_name = {}
    for line in p['smt']:
        tag = re.search(r':named (\S+?)\)', line)
        if tag:
            by_name[tag.group(1)] = line.strip()
    print(f"{p['pair_id']}: INELIGIBLE -- {len(p['violated'])} requirement(s) violated")
    for name in p['violated']:
        body = by_name.get(name, '')
        shown = re.sub(r'\s+', ' ', readable(body))
        print(f'   {name}')
        print(f'      {shown}')
    return p['violated']

violated = pivotal(pair)


sigir-20144__NCT02114099: INELIGIBLE -- 2 requirement(s) violated
   REQ0_COMPONENT1_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      patient_has_diagnosis_of_giant_aneurysm_now
   REQ1_COMPONENT0_OTHER_REQUIREMENTS
      (> patient_age_value_recorded_now_in_years 10)


### 4. Self-faithfulness — does the verdict actually depend on that?

So far, the pivotal conditions are the system's *claim* about why it made the
decision.
To test whether that claim is true, you cannot just reread the explanation. You
have to:

> **change what the system says the decision depends on, re-run it, and see
> whether the answer changes.**

Here, remove every pivotal condition and solve again. If the verdict is still
`INELIGIBLE`, those conditions do not account for the decision. For the aneurysm
trial, removing the age and aneurysm requirements leaves only the Kawasaki
requirement, which the patient meets, so the verdict becomes `ELIGIBLE`.

For a constraint system, this can be tested by re-solving. We re-solved
with Z3 when preparing the data; the cell below reads the result.

In [54]:
# Retract every pivotal condition, then read the re-solve (done with Z3
# when the data was prepared -- this notebook does not run a solver).
def retract_pivotal(p):
    """Remove every violated requirement; did the verdict flip?"""
    p = pair_of(p)
    if p['eligible']:
        print(f"{p['pair_id']}: eligible already -- nothing to retract.")
        return None
    flipped = p['flips_after_retraction']
    print(f"{p['pair_id']}  (verdict was INELIGIBLE)")
    print(f"   pivotal conditions retracted : {len(p['violated'])}")
    print(f"   verdict after re-solving     : {'ELIGIBLE -- it flipped' if flipped else 'still INELIGIBLE'}")
    return flipped

retract_pivotal(pair)
print()
inel = [q for q in PAIRS if not q['eligible']]
with contextlib.redirect_stdout(io.StringIO()):
    n = sum(bool(retract_pivotal(q)) for q in inel)
print(f'across all {len(inel)} ineligible demo pairs: {n} flip.')


sigir-20144__NCT02114099  (verdict was INELIGIBLE)
   pivotal conditions retracted : 2
   verdict after re-solving     : ELIGIBLE -- it flipped

across all 7 ineligible demo pairs: 7 flip.


**What this shows.** Every pair flips. That is expected: once every violated
requirement is gone, what's left can all hold for the patient, and for a
constraint system the verdict follows directly from the constraints. The full
version, run on a modified chart, is
[`examples/01_decide_a_pair.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/01_decide_a_pair.py).

**Why run it anyway.** The same experiment works on systems where the outcome
is not a foregone conclusion. For a matcher that explains itself in a paragraph, you edit
the chart to remove the reasons it gave, ask the model again, and check whether
the answer changes. Published results find that free-text matchers often do
*not* change: they give a reason, but their answer remains unchanged after its
removal, so the stated
reason may not account for the answer.

Two limits of this test:

- **Passing it is not the same as being right.** A system that always answered
  "ineligible, because the patient is under 18" would flip perfectly whenever
  age changed, yet its decisions could still be wrong. Flipping provides
  evidence that the stated reasons
  affect the system's answer; it says nothing about whether they are *good*
  reasons.
- **Our setting differs from testing a model.** Re-running a solver does not require a
  model call and gives the
  same answer for the same input. Doing this with a model costs one call per
  edited
  chart, requires a way to make each edit and a way to check that the edit
  changed only what it should, and systems tested this way may score below
  perfect performance.

> **TODO 3.** Reflect:
>
> **(a)** Of the four things in this part — policy, assumptions, pivotal
> conditions, self-faithfulness — which would you most want if you were the
> coordinator who has to phone the patient? Which would you most want if you were
> the regulator auditing the service a year later? Say why they differ, if they
> do.
>
> **(b)** Here the policy is a written document, fixed in advance. The
> alternative is to let a model judge each chart on its merits. Make the case for
> **both**: what is good and what is bad about each approach? Then say which you
> would use, and when.

*Where this is implemented:* The audit artifacts live in
[`verdict/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/verdict)
and
[`audit/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/audit),
and
[`examples/02_read_the_artifacts.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/02_read_the_artifacts.py)
prints them for one pair. The chart-editing experiment for model-based matchers
is in
[`counterfactual_modifier/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/counterfactual_modifier).
The VERDICT paper is
[arxiv.org/abs/2609.03366](https://arxiv.org/abs/2609.03366).

In [58]:
# YOUR TURN (TODO 3) -- the audit tools, on any pair you like.
#
#   verdict(p) / evidence_states(p) / pivotal(p) / retract_pivotal(p)

for p in PAIRS:
    if p['unsat_core']:
        pivotal(p)
        print()
        break


sigir-201524__NCT00929721: INELIGIBLE -- 1 requirement(s) violated
   REQ0_COMPONENT0_PRESCREEN_NOTES_MUST_COMPLETELY_SUFFICE
      (>= patient_age_value_recorded_now_in_years 50.0)



In [59]:
# YOUR ANSWERS for TODO 3. Replace the placeholder text, then run the cell.
# Keep the triple quotes. Running it again overwrites the previous answer.

todo_3a = """
The **clinical trial coordinator** phoning the patient and the **regulator** auditing the service a year later have fundamentally different operational goals—one is making a *case-level, actionable screening decision right now*, while the other is evaluating *system-level governance, calibration, and explanation integrity across a population*.

**1. What the Coordinator Most Wants: Assumptions + Pivotal Conditions**
- **If the verdict is `ELIGIBLE`:** The coordinator most wants **Assumptions (and Deferred items)** (`evidence_states(p)` / `basis(p)`, Cells 22 & 48). A bare `ELIGIBLE` label hides whether the patient matched on documented chart facts (like `sigir-20144__NCT00000520`, where 4/4 variables come from `CHART`) or on unrecorded `UNKNOWN_DEFER` / `ASSUMED_FALSE` defaults (like `sigir-201522__NCT01259336c`, where 3/6 variables are `POLICY: UNKNOWN_DEFER`, or `sigir-20141__NCT01084239`, where 31/48 variables rest on policy rules). Before phoning the patient, the coordinator needs the exact checklist of unrecorded items to ask on the call.
- **If the verdict is `INELIGIBLE`:** The coordinator most wants **Pivotal Conditions** (`pivotal(p)`, Cell 51)—not the raw Z3 `unsat_core`. As shown in `sigir-20144__NCT02114099` (Cells 19 vs. 51) and `sigir-201524__NCT00929721` (Cell 57), the raw `unsat_core` includes requirements the patient actually *meets* (e.g., `REQ0_COMPONENT0` for Kawasaki disease in `NCT02114099`, or 3 requirements in `NCT00929721` when only `REQ0_COMPONENT0` `(>= age 50.0)` is violated). Pivotal conditions tell the coordinator the exact minimal set of violated requirements—and critically, whether a violation rests on a hard chart fact (`age = 2.0` vs. `> 10`, which cannot be fixed by a phone call) or a policy assumption (`giant_aneurysm = assumed_false`, which might be overturned if an outside echocardiogram report exists).

**2. What the Regulator Most Wants: Policy + Self-Faithfulness**
- **The Written Policy (`POLICY`, Cell 46):** A regulator auditing the service a year later needs to know that missingness was handled by an explicit, versioned, clinically justified rule table (29 rules across categories like `1a common chronic disease -> unknown_defer` vs. `1b named specific disease -> assumed_false`) rather than arbitrary, non-reproducible LLM hallucinations that vary across patients, demographics, or prompt runs.
- **Self-Faithfulness (`retract_pivotal(p)`, Cells 52–55):** Regulators must verify that the explanations shown to clinicians are **causally faithful** to the decision mechanism, not post-hoc rationalizations. While deterministic constraint solvers flip 7/7 ineligible demo pairs upon retracting pivotal conditions (Cell 53), free-text LLM matchers frequently fail counterfactual chart-editing tests (continuing to output "ineligible" even after their stated reason is removed from the chart).

**Why they differ:** The coordinator needs *patient-specific actionable deltas* (what is missing from this chart, or what specific barrier excluded this patient) to conduct a 5-minute screening call; the regulator needs *institutional guarantees of consistency, non-arbitrariness, and causal faithfulness* across thousands of historical decisions.
"""

todo_3b = """
**1. Fixed Written Policy (VERDICT's approach in Cells 45–48)**
- **Strengths (What is good):**
  - *Determinism, Reproducibility, and Auditability:* Every silent chart predicate in a given clinical category (e.g., `Uncommon Chronic Disease` $\rightarrow$ `assumed_false`; `Past or Ongoing Diagnostic Procedure` $\rightarrow$ `unknown_defer`) is resolved identically across all 3,621 trials and 59 patients. Every assumption is explicitly logged in `evidence_states(p)` with its category and rationale.
  - *Zero Marginal Inference Cost & Calibration Control:* Applying the 29-rule lookup table in `policy_value(var)` requires zero LLM calls and allows clinical governance boards to tune sensitivity vs. specificity globally (e.g., changing a category from `assumed_false` to `unknown_defer`).
- **Weaknesses (What is bad):**
  - *Context-Blindness to Note Type and Clinical Implication:* A fixed rule table only sees the predicate's category (`CATEGORY[var]['level2']`), not the clinical context of the note. For example, in `sigir-20141__NCT01084239` (*Multicenter Study to Rule Out Myocardial Infarction by Cardiac CT*), `patient_has_finding_of_sinus_rhythm_now` is classified as `Symptom/Sign / Exam Finding (Current Visit)` where silence maps to `ASSUMED_FALSE`. Because the brief ED note did not explicitly state "sinus rhythm", the fixed policy assumes the patient is *not* in sinus rhythm (`assumed_false`), making `REQ5_COMPONENT0_OTHER_REQUIREMENTS` one of the two pivotal violations (`Cell 51`)! A clinician reading a normal heart rate/ECG note without arrhythmia mentioned would infer sinus rhythm is likely present or at least defer (`unknown_defer`), whereas the rigid policy rule falsely disqualifies the patient.

**2. Model Judgment on Each Chart's Merits (LLM-as-Judge / Free-Text Matcher)**
- **Strengths (What is good):**
  - *Contextual Nuance and Pragmatic Clinical Inference:* A language model reading the full note can condition on note length, care setting (brief ED triage vignette vs. comprehensive longitudinal EHR), and correlated findings (e.g., recognizing that an uncomplicated vitals line in a chest-pain vignette shouldn't be treated as proof of arrhythmia/non-sinus rhythm, or inferring from medications/labs that a chronic condition is absent).
- **Weaknesses (What is bad):**
  - *Silent, Unaudited Assumptions & Run-to-Run Variance:* When an LLM judges silence on the fly, it embeds unstated assumptions inside prose rationales, may treat the exact same silent predicate as `False` for Patient A and `True` for Patient B, and often fails self-faithfulness (Cells 54–55).

**Which I would use, and when:**
I would use a **Fixed Written Policy as the default backbone** for high-throughput clinical trial screening, regulatory auditing, and formal SMT solving (`SatIR`/`VERDICT`), because silent assumptions must be visible and reproducible. However, I would pair it with **targeted Model Judgment as an exception/escalation layer** strictly for predicates marked `unknown_defer` or `assumed_false` that end up in `pivotal(p)` (like `sinus_rhythm_now` in `sigir-20141__NCT01084239`): when a policy assumption alone causes an `INELIGIBLE` verdict, a second contextual model check (or coordinator flag) can estimate the posterior probability from the note context without sacrificing the solver's formal audit trail.
"""

print('TODO 3: recorded 2 answer(s).')


TODO 3: recorded 2 answer(s).


---
### Summary: who did what

| step | language model | code |
|---|---|---|
| read criteria, name the variables | ✅ | |
| write the assertions over those variables | ✅ | |
| read the patient's chart against those variables | ✅ | |
| decide sat / unsat | | ✅ |
| project atoms into tables | | ✅ |
| retrieve across 3,621 trials | | ✅ |
| apply the policy on silence | | ✅ |
| list assumptions | | ✅ |
| find the unsat core and the pivotal conditions | | ✅ |
| retract and re-solve | | ✅ |

The model owns the top three rows and nothing else. Each row is a job done by a
chain of calls: the trial side happens offline, the patient read happens once
per pair. Nothing below those rows calls a model. That is what makes Parts 2 and
3 possible: code can be re-run and can say why it decided.

The cost is that the code only knows what the translation gave it. If the model
translated a criterion wrongly, downstream results may reflect that translation
error. Part 4
asks how this approach could be combined with GRILL from Homework 1, where the
criteria are not written down in advance.

---
# Part 4 — Combining the two

The tasks above used **criteria that were written down**. A trial
sponsor wrote "FEV1 < 50% of predicted", and the model's job was translation.

Homework 1 gave you the opposite situation. A rare disease, a cohort of patients
described by HPO terms (codes from the Human Phenotype Ontology, a standard list
of symptoms and findings), exactly one of whom has it — and no provided document
stating what the disease requires. GRILL had to go and find that out.

The two systems address different aspects of the same problem.

### The two halves

**GRILL finds out what the criteria might be.** It searches the literature and
keeps a belief ledger — a running record, only ever added to, holding the
hypotheses it is entertaining, the evidence for and against each, and the
questions it has not answered. When a hypothesis is refuted it becomes a new
question rather than being deleted. What it produces is a report and a ranking
of the cohort.

**SatIR and VERDICT decide, and show their work.** Given criteria already in
hand, Parts 1-3 turned them into constraints, retrieved against them at scale,
and made the decision inspectable in four ways:

| | what it gives you |
|---|---|
| **policy** | a written-down rule for what a silent record means, per kind of finding |
| **assumptions** | the list of values the system supplied that nobody recorded |
| **pivotal conditions** | every requirement the patient violates; all must be fixed for the verdict to flip |
| **self-faithfulness** | change those requirements, re-run, see whether the answer moves |

Neither half is the whole system. GRILL can propose criteria but has no formal
account of what it assumed or what its ranking rests on. SatIR and VERDICT can
give that account, but only once somebody hands them the criteria.

> **TODO 4.** Open question, argued rather than computed. In roughly 400 words:
>
> **How would you combine GRILL with SatIR/VERDICT to do better at rare-disease
> diagnosis than GRILL alone?** Sketch the combined system, then give at least
> two **concrete** examples of a case it would get right that GRILL by itself
> would not, or an error it would catch that GRILL would miss.
>
> Then name **one or two specific challenges** that would make it hard to build,
> and work each through a concrete example: what goes wrong, on what kind of
> case, and how you might get around it.
>
> There is no expected answer. Be specific about mechanism.
>
> Threads worth pulling, if you want a start:
>
> - Homework 1's own example: the report treats "hypotonia" as typically
>   present, but the top-ranked patient's record notes its **absence**. Which
>   part of Part 3 is about exactly that?
> - A record lists the HPO terms a patient *has*. What does a term's absence
>   mean, and which of the five evidence states would you give it?
> - Homework 1 asked what separates rank 1 from rank 2. What would the pivotal
>   conditions give you there?
> - Homework 1 asked whether repeated runs agreed. What does the Part 3
>   self-faithfulness probe add to that?
> - What does `UNSAT` mean when the constraints are themselves a hypothesis?

In [64]:
# YOUR ANSWER for TODO 4. Replace the placeholder text, then run the cell.
# Keep the triple quotes. Running it again overwrites the previous answer.

todo_4 = """
### Sketch of the Combined Architecture: **GRILL-VERDICT**

In Homework 1, **GRILL** discovers candidate disease criteria from literature via an append-only belief ledger (`directions`, `hypotheses`, `evidence`), and **SLIDERS** (HW2 Part 1) distills the resulting report into structured phenotype tables—yet the final cohort ranking still relies on an uncalibrated LLM prompt (`report_match.json`) that weighs `for` and `against` arguments in free text. We can combine **GRILL** with **SatIR/VERDICT** in a four-stage neuro-symbolic pipeline:

1. **Literature-to-SMT Disease Compiler (GRILL $\rightarrow$ Step 1/2 Compiler):**
   Instead of stopping at a prose `answer.md`, each `pooled` (supported) hypothesis in GRILL's `ledger.json` is compiled into formal SMT declarations and named assertions over canonical **HPO terms**, inheritance modes, age-of-onset intervals, and genetic loci—annotated with literature penetrance weights (e.g., obligate $100\%$, very frequent $80\text{--}99\%$, occasional $5\text{--}29\%$).
2. **Five-State Phenopacket Evidence Policy (VERDICT Policy + Assumptions):**
   Each patient in `cohort.json` has `terms` (observed HPO codes) and `excluded` (explicitly absent HPO codes). We map these directly to VERDICT's five evidence states (Cell 46):
   - $t \in \text{patient.terms}$ (or HPO ancestor via subsumption) $\rightarrow$ **`explicit_true`**
   - $t \in \text{patient.excluded}$ $\rightarrow$ **`explicit_false`**
   - $t$ unmentioned ($t \notin \text{terms} \cup \text{excluded}$) $\rightarrow$ governed by an **HPO-aware silence policy**: highly salient/obligate structural malformations default to `assumed_false`, whereas subtle, age-dependent, or lab-specific phenotypes default to **`unknown_defer`**.
3. **MaxSMT / Weighted Solver &VERDICT Audit (`pivotal` + `retract_pivotal`):**
   An SMT/MaxSMT solver checks each of the 10 cohort patients against the compiled disease constraints, producing formal **pivotal conditions** (which obligate/high-frequency disease features a patient explicitly violates) and **self-faithfulness checks** (whether retracting a single contested literature hypothesis flips the top-ranked patient).
4. **Closed-Loop Corrective Directions (`UNSAT` $\rightarrow$ GRILL Frontier):**
   Because the disease constraints are themselves *hypotheses*, if all 10 cohort patients return `UNSAT` (or if the top candidates tie), the solver's pivotal conditions automatically spawn a **corrective direction** in GRILL's ledger to query the literature specifically on the discriminating or violated phenotype!

---

### Two Concrete Examples Where GRILL-VERDICT Beats GRILL Alone

1. **Catching Explicit Negatives (`explicit_false` vs. Silence) — The Hypotonia & Core Feature Trap:**
   In Homework 1 (e.g., `OMIM:620851` ReNU syndrome / `OMIM:620450` *CHDFIDD*), GRILL's prose matcher (`report_match.json`) frequently ranks a distractor patient #1 or #2 even when that patient's chart records an **explicit negative** (`excluded`) for a hallmark feature (such as hypotonia `HP:0001252`, microcephaly `HP:0000252`, or global developmental delay `HP:0001263`), or conflates "not mentioned in `terms`" with "`excluded`". Under GRILL-VERDICT, an HPO term in `patient.excluded` is bound to **`explicit_false`**, whereas an unmentioned term in the true patient is **`unknown_defer`**. When GRILL's report establishes that a hallmark phenotype is present in $>85\%$ of cases, `explicit_false` triggers a formal **pivotal condition violation** in `pivotal(p)` against the distractor patient, whereas `unknown_defer` on the true patient does not violate the constraint.
2. **Eliminating Stochastic Tie-Breaking Between Rank 1 and Rank 2 via Pivotal Conditions & Self-Faithfulness:**
   In Homework 1 (TODO 3.1 & 5.1), running the LLM ranker 3 times on the *exact same* GRILL report (`report_match.json`) produces shifting probabilities and rank swaps between the top two candidates (e.g., `P03` vs. `P07`) because the LLM arbitrarily re-weights overlapping non-specific HPO terms (like seizures or intellectual disability). In GRILL-VERDICT, `pivotal(p)` deterministically isolates the exact set of requirements separating Rank 1 (`0` violated core criteria) from Rank 2 (`1` violated criterion, e.g., onset age or an excluded dysmorphic feature), and `retract_pivotal(p)` proves whether the ranking rests on a single low-confidence hypothesis in `ledger.json`.

---

### Two Specific Technical Challenges and Concrete Workarounds

1. **Challenge 1: Incomplete Penetrance and Variable Expressivity (`UNSAT` Under Strict Boolean Logic):**
   *What goes wrong:* Clinical trial criteria are hard legal rules ("must be $>10$ years old"), whereas rare diseases exhibit **incomplete penetrance** (e.g., in *NEDAMSS* or *ReNU syndrome*, seizures or optic atrophy occur in $40\text{--}70\%$ of patients, not $100\%$). If Step 2 compiles every phenotype in GRILL's report as a hard `(assert patient_has_HP_...)`, a true patient who happens to carry `explicit_false` (`excluded`) for a $60\%$-penetrant symptom will return `UNSAT`!
   *Workaround:* Compile only obligate ($100\%$) features as hard `(assert ...)` constraints, and compile variable-penetrance features as **weighted soft constraints (`assert-soft` in Z3 MaxSMT)** with weights derived from the cohort frequencies GRILL extracted (e.g., $\log \frac{p}{1-p}$), reserving hard `UNSAT` for pathognomonic/obligate criteria or contradictory inheritance modes.
2. **Challenge 2: HPO Ontology Granularity Mismatch (Canonicalization Across Hierarchy Levels):**
   *What goes wrong:* As seen in SatIR's `@@` refinement dropping (Cell 30), if GRILL's literature compiler declares `patient_has_generalized_tonic_clonic_seizures` (`HP:0002069`) while the patient's phenopacket records `Seizure` (`HP:0001250`) or `Focal impaired awareness seizure` (`HP:0002384`), a flat string join treats the chart as silent!
   *Workaround:* Embed **HPO DAG subsumption axioms** as `AUTO_AUXILIARY` implications in the SMT program—just like `AUTO_AUXILIARY_QUAL_IMPLIES_STEM_0` in `sigir-201522__NCT01259336c` (Cell 22)—so `(assert (=> child_HPO_term parent_HPO_term))` propagates `explicit_true` upward and `explicit_false` downward across the HPO hierarchy.
"""

print('TODO 4: recorded.')


TODO 4: recorded.


In [65]:
# Save every answer you wrote above into one file to hand in.
import json

answers = {k: v.strip() for k, v in sorted(globals().items())
           if k.startswith('todo_') and isinstance(v, str)}
blank = [k for k, v in answers.items() if v.startswith('(your answer')]

with open('todo_answers.json', 'w') as f:
    json.dump(answers, f, indent=1)

print(f'wrote todo_answers.json with {len(answers)} answers')
if blank:
    print(f'still on the placeholder text: {", ".join(blank)}')
else:
    print('none are still on the placeholder text.')


wrote todo_answers.json with 6 answers
none are still on the placeholder text.


### Where the real code lives

Everything in this notebook came from a real system, and it is open source:

**https://github.com/stanford-oval/clinical-trial-matching**

It includes runnable examples for each part above. To explore a part further,
start with its example rather than with the module code.

| part | the example to run | the module behind it |
|---|---|---|
| 1 — text → constraints | [`examples/05_the_demo_pair.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/05_the_demo_pair.py) | [`trial_compiler/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/trial_compiler), [`patient_compiler/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/patient_compiler) |
| 1 — solving a pair | [`examples/01_decide_a_pair.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/01_decide_a_pair.py) | [`smt_core/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/smt_core), [`smt_matcher/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/smt_matcher) |
| 2 — retrieval | [`examples/06_satir_retrieval.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/06_satir_retrieval.py) | [`sql_retrieval/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/sql_retrieval), [`db_indexer/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/db_indexer) |
| 2 — screening one patient | [`examples/04_screen_a_patient.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/04_screen_a_patient.py) | `sql_retrieval/ops/` |
| 3 — the audit trail | [`examples/02_read_the_artifacts.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/02_read_the_artifacts.py) | [`verdict/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/verdict), [`audit/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/audit) |
| 3 — self-faithfulness | — | [`counterfactual_modifier/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/counterfactual_modifier) |
| adding your own matcher | [`examples/03_add_your_own_matcher.py`](https://github.com/stanford-oval/clinical-trial-matching/blob/main/examples/03_add_your_own_matcher.py) | [`matchers/`](https://github.com/stanford-oval/clinical-trial-matching/tree/main/matchers) |

The repository and the two papers are linked at the top of this notebook.

The data embedded in this notebook was extracted from real runs of the
trial-matching code over the SIGIR collection (a public benchmark of 59
patient notes and 3,621 clinical trials). Nothing here is synthetic.